In [1]:
import os
import glob
import pandas as pd

In [4]:
import os
import glob
import pandas as pd

# Define o diretório contendo os arquivos Parquet
directory = 'datasets/ts_ndt'

# Lista para armazenar os DataFrames individuais
dfs = []

# Itera sobre todos os arquivos Parquet no diretório
for file in glob.glob(os.path.join(directory, "*.parquet")):
    # Extrai o nome do arquivo e remove a extensão
    base_name = os.path.basename(file)                # Ex: "cliente_servidor.com_extra.parquet"
    name_without_ext = os.path.splitext(base_name)[0]   # Ex: "cliente_servidor.com_extra"
    
    # Divide o nome no primeiro underscore para separar cliente e servidor
    # Assim, 'cliente' é a parte antes do primeiro "_" e 'servidor' é o restante
    client, server = name_without_ext.split('_', 1)
    
    # Lê o arquivo Parquet e adiciona as colunas 'client' e 'server'
    df = pd.read_parquet(file)
    df['client'] = client
    df['server'] = server
    dfs.append(df)

# Concatena todos os DataFrames em um único DataFrame
all_data = pd.concat(dfs, ignore_index=True)

# Seleciona as colunas numéricas, que correspondem às métricas
metrics_columns = all_data.select_dtypes(include='number').columns.tolist()

# Remove as colunas de agrupamento, caso por acaso estejam na lista
for col in ['client', 'server']:
    if col in metrics_columns:
        metrics_columns.remove(col)

# Cálculo das estatísticas para cada cliente
client_stats = (
    all_data
    .groupby("client")[metrics_columns]
    .agg(["mean", "std"])
    .reset_index()
)

# Cálculo das estatísticas para cada servidor
server_stats = (
    all_data
    .groupby("server")[metrics_columns]
    .agg(["mean", "std"])
    .reset_index()
)

# Exibe os DataFrames com as estatísticas calculadas
print("Estatísticas por Cliente:")
print(client_stats)

print("\nEstatísticas por Servidor:")
print(server_stats)


Estatísticas por Cliente:
      client rtt_download            throughput_download              \
                     mean        std                mean         std   
0   client01     9.669518   5.313520          515.127084   23.925863   
1   client02    13.917631   4.125960          586.034211   23.835895   
2   client03    52.746522  74.159755          592.675817  213.083369   
3   client04     9.736546   5.602429          741.381987  134.824215   
4   client05    24.100288  24.381113          567.069542  166.571677   
5   client06    44.050972  49.884659          629.080750  331.805103   
6   client07    29.890629  37.010311          712.972492  277.812186   
7   client08    35.674549  36.145701          649.097691  244.206697   
8   client09    28.548239  35.627153          718.562621  271.749191   
9   client10    32.343269  36.864353          739.954085  249.246338   
10  client11    31.369263  34.217914          705.366589  218.816884   
11  client12    31.312221  37.826034  

In [5]:
# Renomeia as colunas de agrupamento para um identificador comum e adiciona uma coluna para indicar o grupo
client_stats = client_stats.rename(columns={'client': 'id'})
client_stats['group'] = 'client'

server_stats = server_stats.rename(columns={'server': 'id'})
server_stats['group'] = 'server'

# Junta os DataFrames de clientes e servidores em um único DataFrame
combined_stats = pd.concat([client_stats, server_stats], ignore_index=True)

# Exibe o DataFrame combinado com as estatísticas agregadas
print("Estatísticas agregadas:")
print(combined_stats)

Estatísticas agregadas:
          id rtt_download            throughput_download              \
                     mean        std                mean         std   
0   client01     9.669518   5.313520          515.127084   23.925863   
1   client02    13.917631   4.125960          586.034211   23.835895   
2   client03    52.746522  74.159755          592.675817  213.083369   
3   client04     9.736546   5.602429          741.381987  134.824215   
4   client05    24.100288  24.381113          567.069542  166.571677   
5   client06    44.050972  49.884659          629.080750  331.805103   
6   client07    29.890629  37.010311          712.972492  277.812186   
7   client08    35.674549  36.145701          649.097691  244.206697   
8   client09    28.548239  35.627153          718.562621  271.749191   
9   client10    32.343269  36.864353          739.954085  249.246338   
10  client11    31.369263  34.217914          705.366589  218.816884   
11  client12    31.312221  37.826034    

In [6]:
combined_stats.to_csv('general_statistics.csv', index=False)